# Nemotron-3 Ultra Text2SQL LoRA Fine-Tuning using Megatron Bridge

This notebook walks you through **data prep → checkpoint conversion → LoRA fine-tuning** of Nemotron-3 Ultra for the Text2SQL use case with [NeMo Megatron-Bridge](https://github.com/NVIDIA-NeMo/Megatron-Bridge).

Nemotron-3 Ultra is a **550B-total / A55B-active hybrid Mamba-Transformer MoE** model. This notebook **does not fit on one node**, so every heavy step runs as a **multi-node SLURM job**. This notebook is meant to be run **from a cluster login node** (where `sbatch`/`squeue` are available); each step is a *launch* cell, a re-runnable *check* cell, and a *sanity* cell.

## Prerequisites

- Access to a SLURM cluster with **GB200** nodes (this tutorial assumes 4 GPUs/node) and Pyxis/enroot (so you can launch `srun --container-image=...`).
- The **Nemotron-3 Ultra checkpoint** already downloaded to a shared path — `HF_MODEL_PATH` below.
- A **Hugging Face token** placed at `${HF_HOME}/token`.
- This cookbook directory on a **shared filesystem** the compute nodes can mount.
- A working **container image** (see the placeholder in the setup cell).

---
## Environment & SLURM Setup

**This is the one place to configure everything** — node counts, account/partition/QOS, paths, model, and hyperparameters. Edit the values below and run the cell; it writes `config.env`, which every step (and the backing `*.sbatch` scripts) reads.

In particular, **`WORKSPACE` is the single output root** — set it once and everything the tutorial produces lands under it in a clean layout: `$WORKSPACE/{base, dataprep, trained, cache/hf, logs}` (converted checkpoint, data, LoRA adapters, HF cache, and SLURM logs).

In [ ]:
import os

COOKBOOK_DIR = os.getcwd()  # this notebook's directory (must be on a shared FS the compute nodes can mount)

# ============================================================================
# EDIT THESE for your cluster. (Ask your admin, or see your environment reference.)
# ============================================================================
CONTAINER_IMAGE = "<CONTAINER_IMAGE>"     # Ultra container: NeMo 26.04.01
ACCOUNT         = "<SLURM_ACCOUNT>"
PARTITION       = "<GPU_PARTITION>"       # GPU partition for the 12-node convert/train jobs
QOS             = "<GPU_QOS>"             # a QOS that allows multi-node GPU jobs
CPU_PARTITION   = "<CPU_PARTITION>"       # CPU partition for dataprep
CPU_QOS         = "<CPU_QOS>"
GPUS_PER_NODE   = 4                       # GPUs per each of your nodes (e.g. GB200 NVL72 = 4 GPUs/node)

# Shared paths (e.g. lustre) that is visible to all nodes running the job:
WORKSPACE       = "<WORKSPACE>"           # OUTPUT root: everything generated lands under here
HF_MODEL_PATH   = "<HF_MODEL_PATH>"       # INPUT (read-only): pre-downloaded Ultra base checkpoint
STORAGE_ROOT     = "<STORAGE_ROOT>"         # bind-mounted into the container; must contain WORKSPACE and HF_MODEL_PATH
# Put your Hugging Face token at ${WORKSPACE}/cache/hf/token (used to download BIRD during data prep).

# ---- Tunables (sensible defaults; fine to leave as-is) ----
MAX_SEQ_LEN       = 4096
MAX_TRAIN_SAMPLES = 0        # 0 = use all samples. You can use small positive N for a short smoke run
GLOBAL_BS         = 32
MICRO_BS          = 1
EPOCHS            = 1
LR                = "1e-4"
TIME_DATAPREP     = "01:00:00"  # SLURM submission time limit for dataprep (single-node CPU job)
TIME_CONVERT      = "02:00:00"  # SLURM submission time limit for the 12-node convert job (resharding the checkpoint for TP/PP settings)
TIME_TRAIN        = "02:00:00"  # SLURM submission time limit for the 12-node training job

# Nemotron-3 Ultra needs AT LEAST 48 GPUs at these parallel settings (convert TP1·PP6·EP8, train TP2·PP6·EP8;
# only TP differs, so torch_dist reshards on load). Node count derives from your GPUs/node.
WORLD_GPUS    = 48
NODES_CONVERT = WORLD_GPUS // GPUS_PER_NODE
NODES_TRAIN   = WORLD_GPUS // GPUS_PER_NODE
CONV  = dict(TP=1, PP=6, EP=8, ETP=1)
TRAIN = dict(TP=2, PP=6, EP=8, ETP=1, CP=1, SP="True")
EXPERIMENT_NAME = "ultra-text2sql-lora"

# Guard: don't let an unfilled placeholder silently launch a broken 12-node job.
for _k, _v in {"CONTAINER_IMAGE": CONTAINER_IMAGE, "ACCOUNT": ACCOUNT, "PARTITION": PARTITION,
               "QOS": QOS, "CPU_PARTITION": CPU_PARTITION, "CPU_QOS": CPU_QOS,
               "WORKSPACE": WORKSPACE, "HF_MODEL_PATH": HF_MODEL_PATH, "STORAGE_ROOT": STORAGE_ROOT}.items():
    assert not str(_v).startswith("<"), f"Set {_k} above (still a placeholder)."

# ---- Output layout: everything under WORKSPACE ----
#   $WORKSPACE/{base (Step 2), dataprep (Step 1), trained (Step 3), cache/hf, logs}
config = f'''\
CONTAINER_IMAGE={CONTAINER_IMAGE}
WORKDIR=/workspace/Megatron-Bridge
COOKBOOK_DIR={COOKBOOK_DIR}
WORKSPACE={WORKSPACE}
LOG_DIR={WORKSPACE}/logs
CONTAINER_MOUNTS={STORAGE_ROOT}:{STORAGE_ROOT}
HF_MODEL_PATH={HF_MODEL_PATH}
MEGATRON_MODEL_PATH={WORKSPACE}/base
DATAPREP_OUTPUT_DIR={WORKSPACE}/dataprep
DATASET_DIR={WORKSPACE}/dataprep
TRAIN_OUTPUT_DIR={WORKSPACE}/trained
EXPERIMENT_NAME={EXPERIMENT_NAME}
HF_HOME={WORKSPACE}/cache/hf
MAX_SEQ_LEN={MAX_SEQ_LEN}
MAX_TRAIN_SAMPLES={MAX_TRAIN_SAMPLES}
ACCOUNT={ACCOUNT}
GPUS_PER_NODE={GPUS_PER_NODE}
PARTITION={PARTITION}
QOS={QOS}
NODES_CONVERT={NODES_CONVERT}
NODES_TRAIN={NODES_TRAIN}
TIME_CONVERT={TIME_CONVERT}
TIME_TRAIN={TIME_TRAIN}
CPU_PARTITION={CPU_PARTITION}
CPU_QOS={CPU_QOS}
TIME_DATAPREP={TIME_DATAPREP}
CONV_TP={CONV['TP']}
CONV_PP={CONV['PP']}
CONV_EP={CONV['EP']}
CONV_ETP={CONV['ETP']}
TRAIN_TP={TRAIN['TP']}
TRAIN_PP={TRAIN['PP']}
TRAIN_EP={TRAIN['EP']}
TRAIN_ETP={TRAIN['ETP']}
TRAIN_CP={TRAIN['CP']}
TRAIN_SP={TRAIN['SP']}
GLOBAL_BS={GLOBAL_BS}
MICRO_BS={MICRO_BS}
EPOCHS={EPOCHS}
LR={LR}
MIN_LR=1e-5
EVAL_ITERS=2
WANDB_MODE=disabled
'''

os.makedirs(os.path.join(WORKSPACE, "logs"), exist_ok=True)
with open(os.path.join(COOKBOOK_DIR, "config.env"), "w") as f:
    f.write(config)
print(config)
print("Wrote", os.path.join(COOKBOOK_DIR, "config.env"))

---
## Step 1 — Data Preparation

Build a BIRD Text2SQL `training.jsonl` from both the no-reasoning and reasoning splits using the Ultra tokenizer/chat-template. By default this uses the **full** BIRD dataset (`MAX_TRAIN_SAMPLES=0`); set `MAX_TRAIN_SAMPLES` to a small positive N for a quick smoke run. Runs as a short CPU job in the container.

### Submit job

In [ ]:
%%bash
set -euo pipefail
source config.env
mkdir -p "$LOG_DIR"
JID=$(sbatch --parsable \
  --account="$ACCOUNT" --partition="$CPU_PARTITION" --qos="$CPU_QOS" \
  --time="$TIME_DATAPREP" --nodes=1 --cpus-per-task=16 \
  --output="$LOG_DIR/dataprep_%j.log" \
  --export=ALL,MBRIDGE_CONFIG="$PWD/config.env" \
  slurm/dataprep.sbatch)
echo "$JID" > .jobid_dataprep
echo "Submitted dataprep job $JID  (log: $LOG_DIR/dataprep_$JID.log)"

### Check status

In [ ]:
%%bash
# Re-run this cell to check status.
JID=$(cat .jobid_dataprep)
sacct -j "$JID" --format=JobID%14,JobName%24,State,Elapsed -n | grep -vE '\.(ba|ex|[0-9])' || true
squeue -j "$JID" 2>/dev/null || true

### Sanity check the results

In [ ]:
%%bash
# Sanity check: training.jsonl exists, has rows, and looks right.
source config.env
F="$DATAPREP_OUTPUT_DIR/training.jsonl"
test -s "$F" || { echo "MISSING/empty: $F (is the dataprep job done?)"; exit 1; }
echo "rows: $(wc -l < "$F")"
echo "--- sample ---"; head -c 600 "$F"; echo

---
## Step 2 — Convert HF → Megatron-Bridge

Distributed import of the 550B base checkpoint (CPU import is infeasible at this size). Uses `NODES_CONVERT` nodes with `TP1·PP6·EP8`. One-time; ~17 min on 12 nodes.

### Submit job

In [ ]:
%%bash
set -euo pipefail
source config.env
mkdir -p "$LOG_DIR"
JID=$(sbatch --parsable \
  --account="$ACCOUNT" --partition="$PARTITION" --qos="$QOS" --time="$TIME_CONVERT" \
  --nodes="$NODES_CONVERT" --gpus-per-node="$GPUS_PER_NODE" --ntasks-per-node="$GPUS_PER_NODE" \
  --output="$LOG_DIR/convert_%j.log" \
  --export=ALL,MBRIDGE_CONFIG="$PWD/config.env" \
  slurm/convert.sbatch)
echo "$JID" > .jobid_convert
echo "Submitted convert job $JID  (log: $LOG_DIR/convert_$JID.log)"

### Check status

In [ ]:
%%bash
# Re-run to check status.
JID=$(cat .jobid_convert)
sacct -j "$JID" --format=JobID%14,JobName%24,State,Elapsed,NNodes -n | grep -vE '\.(ba|ex|[0-9])' || true
squeue -j "$JID" 2>/dev/null || true

### Sanity check the results

In [ ]:
%%bash
# Sanity check: a Megatron checkpoint was written.
source config.env
test -e "$MEGATRON_MODEL_PATH/latest_checkpointed_iteration.txt" || { echo "No converted checkpoint yet at $MEGATRON_MODEL_PATH"; exit 1; }
echo "Converted base checkpoint OK at: $MEGATRON_MODEL_PATH"
ls "$MEGATRON_MODEL_PATH" | head

---
## Step 3 — LoRA Fine-Tuning

LoRA fine-tune on the prepared Text2SQL data using `NODES_TRAIN` nodes with `TP2·PP6·EP8`. The Ultra PEFT recipe supplies the correct model config and LoRA targets (including the Mamba `in_proj`/`out_proj`). Saves a LoRA adapter checkpoint.

Training uses **packed sequences**: rather than padding each example out to the batch's longest member, many examples are concatenated into dense `seq_length`-token blocks. Since Text2SQL examples range from short queries to long reasoning traces, packing avoids the padding waste and covers the same data in far fewer, fully-dense steps — much less GPU time. It's the approach the Megatron-Bridge Ultra recipe is built around.

> **The first iteration is slow! Don't mistake it for a hang.** Before step 1 the job captures CUDA graphs and warms up the MoE, which can take ~15 minutes with no log output and the GPUs pegged at 100%. That's expected; let it run. Subsequent iterations are fast (a few seconds each).

### Submit job

In [ ]:
%%bash
set -euo pipefail
source config.env
mkdir -p "$LOG_DIR"
JID=$(sbatch --parsable \
  --account="$ACCOUNT" --partition="$PARTITION" --qos="$QOS" --time="$TIME_TRAIN" \
  --nodes="$NODES_TRAIN" --gpus-per-node="$GPUS_PER_NODE" --ntasks-per-node="$GPUS_PER_NODE" \
  --output="$LOG_DIR/train_%j.log" \
  --export=ALL,MBRIDGE_CONFIG="$PWD/config.env" \
  slurm/train_lora.sbatch)
echo "$JID" > .jobid_train
echo "Submitted LoRA training job $JID  (log: $LOG_DIR/train_$JID.log)"

### Check status

In [ ]:
%%bash
# Re-run to check status.
JID=$(cat .jobid_train)
sacct -j "$JID" --format=JobID%14,JobName%24,State,Elapsed,NNodes -n | grep -vE '\.(ba|ex|[0-9])' || true
squeue -j "$JID" 2>/dev/null || true

### Sanity check the results

In [ ]:
%%bash
# Sanity check: a LoRA adapter checkpoint was saved.
source config.env
RUN_DIR="$TRAIN_OUTPUT_DIR/$EXPERIMENT_NAME"
ITER_FILE=$(find "$RUN_DIR" -name latest_checkpointed_iteration.txt 2>/dev/null | head -1)
test -n "$ITER_FILE" || { echo "No saved adapter yet under $RUN_DIR"; exit 1; }
echo "Saved adapter checkpoint at iteration: $(cat "$ITER_FILE")"
CKPT_DIR=$(dirname "$ITER_FILE")
echo "Checkpoint dir: $CKPT_DIR"
echo "Contents:"; ls "$CKPT_DIR"

---
## Done

You now have a **trained Nemotron-3 Ultra LoRA adapter** for Text2SQL at `${TRAIN_OUTPUT_DIR}/${EXPERIMENT_NAME}`.